### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_JOBIGCZZUZJO


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250520_squid_cal520"
recname = "rec8"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec8"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(717000,) (10909,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(166, 717000)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.141177,0.130576,0.444809,0.557649,0.329929,-0.040479,-0.268669,-0.073621,...,-0.263941,-0.149434,-0.386434,-0.939480,-0.804915,-0.495590,-0.433565,-0.562757,-0.132278,-0.085032
1,0,1,0.116205,0.465189,0.082054,-0.538819,-0.396493,0.466278,0.501325,-0.009890,...,-0.328999,-0.117907,0.387464,0.467831,-0.048527,-0.302225,-0.034152,-0.230796,0.024268,0.100163
2,0,2,0.159977,0.938181,1.005565,0.361237,1.081253,0.838687,0.524517,0.557357,...,-0.131882,-0.252492,-0.253375,-0.076333,-0.239211,-0.584108,-0.885736,-0.456068,-0.577724,-0.762089
3,0,3,0.314041,0.577875,0.377938,0.959475,1.013815,0.776744,0.427314,0.702734,...,-0.876594,-0.886960,-0.712026,-0.461651,-0.855328,-0.755098,-0.856210,-0.523678,-0.608284,-0.547165
4,0,4,0.352617,-0.349950,-0.452656,-0.553201,-0.238949,-0.309872,-0.493231,-0.565964,...,-0.166235,0.046040,0.340318,0.234635,-0.079176,0.066757,-0.214610,-0.518585,-0.594313,-0.439649
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,0,161,0.481692,0.410814,0.516680,0.493601,0.947505,0.807322,0.887952,0.973214,...,-1.237484,-1.259563,-1.230836,-1.219797,-0.849827,-0.984174,-0.924452,-0.928467,-0.854015,-0.980925
162,0,162,0.304311,0.219007,0.407143,0.427794,0.443195,0.562120,0.066845,0.074034,...,-0.438156,-0.303600,-0.033936,-0.352422,-0.502747,-0.367631,-0.490683,-0.607891,-0.503268,-0.564807
163,0,163,0.408967,-0.260925,-0.371185,-0.454510,-0.418280,-0.657167,-0.822572,-0.335306,...,-1.096922,-1.388931,-1.118495,-0.913441,-0.977300,-1.381582,-1.317008,-1.310372,-0.857343,-0.815292
164,0,164,0.271199,0.218834,0.284557,0.433929,0.565093,0.611279,0.979126,0.941718,...,-0.673666,-0.645455,-0.518847,-0.660062,-0.754519,-0.686808,-0.724576,-0.781733,-0.468953,-0.392528


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec8"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_JOBIGCZZUZJO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec8"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6
Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_JOBIGCZZUZJO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec8"', returncode=0)